# 01 基线核算

目标：把题目中的“功率、时间、PUE、碳强度”变成一个可检查的公式。

边界：这是教学基线，不是全球 HPC 排放估计。

In [ ]:
def carbon_accounting(power_kw=1000, average_power_ratio=0.6, hours=8760, pue=1.2, intensity_kg_per_kwh=0.5):
    """Return annual facility energy and carbon emissions.

    power_kw: IT reference power
    average_power_ratio: average operating power / reference power, 0 to 1
    hours: operating hours
    pue: facility energy / IT energy, >= 1
    intensity_kg_per_kwh: kg CO2e per kWh
    """
    if power_kw < 0:
        raise ValueError('power_kw cannot be negative')
    if not 0 <= average_power_ratio <= 1:
        raise ValueError('average_power_ratio must be between 0 and 1')
    if hours < 0:
        raise ValueError('hours cannot be negative')
    if pue < 1:
        raise ValueError('PUE must be at least 1')
    if intensity_kg_per_kwh < 0:
        raise ValueError('carbon intensity cannot be negative')

    it_energy_kwh = power_kw * average_power_ratio * hours
    facility_energy_kwh = it_energy_kwh * pue
    carbon_t = facility_energy_kwh * intensity_kg_per_kwh / 1000
    return {
        'IT energy kWh': it_energy_kwh,
        'Facility energy kWh': facility_energy_kwh,
        'Carbon tCO2e': carbon_t,
    }

baseline = carbon_accounting()
baseline

In [ ]:
print(f"Baseline carbon: {baseline['Carbon tCO2e']:,.1f} tCO2e/year")
print(f"Facility energy: {baseline['Facility energy kWh']:,.0f} kWh/year")

先预测：如果只把 PUE 从 1.2 改成 1.1，排放下降多少百分比？

In [ ]:
pue_policy = carbon_accounting(pue=1.1)
reduction = 1 - pue_policy['Carbon tCO2e'] / baseline['Carbon tCO2e']
print(f"PUE 1.2 -> 1.1 reduction: {reduction:.2%}")
print(f"Carbon after PUE improvement: {pue_policy['Carbon tCO2e']:,.1f} tCO2e/year")

In [ ]:
low_carbon_power = carbon_accounting(pue=1.1, intensity_kg_per_kwh=0.3)
ratio = low_carbon_power['Carbon tCO2e'] / baseline['Carbon tCO2e']
print(f"PUE=1.1 and intensity=0.3 keeps {ratio:.1%} of baseline carbon")
print(f"Carbon: {low_carbon_power['Carbon tCO2e']:,.1f} tCO2e/year")

论文中要写清楚：百分比的分母是谁，参数改变了哪一个，其他参数是否保持不变。

In [ ]:
# 常见错误演示：PUE 小于 1 不允许。
try:
    carbon_accounting(pue=0.9)
except ValueError as exc:
    print('Caught expected error:', exc)